In [21]:
import concurrent.futures
import dataclasses
import io

import fsspec
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from pacer.external.speedhive import SpeedhiveSession, parse_laptime_seconds

In [7]:
sd_3hr = SpeedhiveSession(session_id=12494589, name="2026-jul-sp-3h")
laptimes = sd_3hr.laptimes()
results_df = sd_3hr.results_df()
results_df

,Start Number,Competitor,Class,Total Time,Diff,Laps,Best Lap,Best Lap No.,Best Speed
1,124,dendi239,DMAX Endurance Championship,0 days 03:00:18.557000,0.000,199,46.906,59,69.074 km/h
2,134,The Mandem - JM,DMAX Endurance Championship,0 days 03:01:02.148000,1 lap,198,46.431,20,69.781 km/h
3,149,WPR,DMAX Endurance Championship,0 days 03:01:02.350000,1 lap,198,46.511,37,69.661 km/h
4,126,Billy No Mates,DMAX Endurance Championship,0 days 03:00:26.683000,3 laps,196,46.940,68,69.024 km/h
5,133,Clean Bulls Racing,DMAX Endurance Championship,0 days 03:00:37.794000,3 laps,196,46.936,134,69.030 km/h
6,128,DRS and Inshallah,DMAX Endurance Championship,0 days 03:00:47.074000,3 laps,196,47.194,10,68.653 km/h
7,144,TDS Racing,DMAX Endurance Championship,0 days 03:01:03.588000,3 laps,196,46.579,142,69.559 km/h
8,148,Maple Motorsport,DMAX Endurance Championship,0 days 03:00:32.546000,4 laps,195,46.438,93,69.770 km/h
9,142,Late Brakers,DMAX Endurance Championship,0 days 03:00:38.013000,4 laps,195,46.920,23,69.054 km/h
10,123,Ferrari LH44,DMAX Endurance Championship,0 days 03:00:31.268000,5 laps,194,46.636,77,69.474 km/h


In [8]:
start_times = results_df.set_index("Competitor")["Total Time"] - laptimes.sum(axis=0)

In [15]:
timestamps = laptimes.assign(**{c: lambda d, c=c: d[c].cumsum() + start_times[c] for c in laptimes.columns})

In [19]:
def plot_laptime_vs_timestamp(
    timestamps: pd.DataFrame, laptimes: pd.DataFrame, results_df: pd.DataFrame
) -> go.Figure:
    ts = timestamps.stack().dt.total_seconds().rename("timestamp_s")
    lt = laptimes.stack().dt.total_seconds().rename("laptime_s")
    df = pd.concat([ts, lt], axis=1).reset_index()
    df.columns = ["lap", "competitor", "timestamp_s", "laptime_s"]

    competitor_to_class = results_df.set_index("Competitor")["Class"]
    df["class"] = df["competitor"].map(competitor_to_class)

    fig = px.scatter(
        df,
        x="timestamp_s",
        y="laptime_s",
        color="class",
        hover_data=["lap", "competitor"],
        log_y=True,
        title="Lap time vs race time - 2026 Jul Daytona SP 3h",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.update_layout(
        xaxis_title="Race time (s)",
        yaxis_title="Lap time (s)",
        legend_title="Class",
    )
    return fig


plot_laptime_vs_timestamp(timestamps, laptimes, results_df)

In [23]:
RACE_STATES = ["green flag lap", "safety kart lap", "pitstop", "safety kart pitstop"]
LAP_STATES = RACE_STATES + ["timing glitch"]


def classify_laps(
    timestamps: pd.DataFrame,
    laptimes: pd.DataFrame,
    *,
    green_quantile: float = 0.3,
    safety_ratio: float = 1.2,
    field_window: str = "150s",
    pit_excess_s: float = 30.0,
    glitch_ratio: float = 0.85,
) -> pd.DataFrame:
    """Label every lap green flag / safety kart / pitstop / safety kart pitstop.

    A safety kart period is a property of the race, a pitstop is a property of one
    competitor: the whole field slows together for the kart, but only a handful of
    teams are stopped at any moment.  The two are therefore detected on different
    signals rather than on lap time alone.

    Each competitor gets their own green reference pace (`green_quantile` of their
    laps), which puts DMAX and SODI on one scale.  The *field median* of
    lap/reference over a rolling `field_window` of race time is then a pace
    multiplier for the race as a whole - flat near 1.0 under green, well above it
    when the kart is out - and pitstops cannot move it, because a median ignores
    the few teams stopped at a time.  Whatever a competitor loses on top of that
    shared multiplier is their own time loss, and more than `pit_excess_s` of it
    is a stop.

    Laps below `glitch_ratio` of the competitor's own reference are impossible to
    drive and are labelled `timing glitch` (a transponder counting one crossing
    twice splits a real lap into fragments), so they are not mistaken for
    exceptionally quick green laps.
    """
    lap_end = timestamps.stack().dt.total_seconds().rename("lap_end_s")
    lap_s = laptimes.stack().dt.total_seconds().rename("laptime_s")
    laps = (
        pd.concat([lap_end, lap_s], axis=1)
        .dropna()
        .rename_axis(index=["lap", "competitor"])
        .reset_index()
        .sort_values("lap_end_s", ignore_index=True)
    )

    green_ref = laptimes.apply(lambda s: s.dt.total_seconds()).quantile(green_quantile)
    laps["green_ref_s"] = laps["competitor"].map(green_ref)
    laps["ratio"] = laps["laptime_s"] / laps["green_ref_s"]

    laps["field_ratio"] = (
        laps["ratio"]
        .set_axis(pd.to_timedelta(laps["lap_end_s"], unit="s"))
        .rolling(field_window, center=True)
        .median()
        .to_numpy()
    )
    laps["expected_s"] = laps["green_ref_s"] * laps["field_ratio"].clip(lower=1.0)
    laps["excess_s"] = laps["laptime_s"] - laps["expected_s"]

    laps["safety"] = laps["field_ratio"] > safety_ratio
    laps["pit"] = laps["excess_s"] > pit_excess_s
    laps["glitch"] = laps["ratio"] < glitch_ratio
    laps["state"] = pd.Categorical(
        np.where(
            laps["glitch"],
            "timing glitch",
            np.where(
                laps["pit"],
                np.where(laps["safety"], "safety kart pitstop", "pitstop"),
                np.where(laps["safety"], "safety kart lap", "green flag lap"),
            ),
        ),
        categories=LAP_STATES,
        ordered=True,
    )
    return laps


def lap_state_summary(laps: pd.DataFrame) -> pd.DataFrame:
    """Table twin of the scatter: how many laps per bucket, and how slow they run."""
    return (
        laps.groupby("state", observed=False)["laptime_s"]
        .agg(laps="size", fastest="min", median="median", slowest="max")
        .assign(share=lambda d: (d["laps"] / d["laps"].sum()).map("{:.1%}".format))
        .round(1)
    )


laps = classify_laps(timestamps, laptimes)
lap_state_summary(laps)

,laps,fastest,median,slowest,share
state,,,,,
green flag lap,6398,46.4,52.9,83.6,88.2%
safety kart lap,577,55.9,88.5,184.1,8.0%
pitstop,217,82.4,140.2,628.8,3.0%
safety kart pitstop,53,132.0,201.4,520.8,0.7%
timing glitch,6,13.7,26.2,37.7,0.1%


In [25]:
laps

,lap,competitor,lap_end_s,laptime_s,green_ref_s,ratio,field_ratio,expected_s,excess_s,safety,pit,glitch,state
0,0,WPR,56.068,55.919,47.0528,1.188431,1.241450,58.413682,-2.494682,True,False,False,safety kart lap
1,0,TDS Racing,56.587,56.040,47.4840,1.180187,1.247110,59.217748,-3.177748,True,False,False,safety kart lap
2,0,The Mandem - JM,57.283,56.471,46.9793,1.202040,1.247110,58.588332,-2.117332,True,False,False,safety kart lap
3,0,Late Brakers,57.706,56.642,47.4980,1.192513,1.252769,59.504040,-2.862040,True,False,False,safety kart lap
4,0,BD Racing,58.598,57.207,47.8298,1.196054,1.266301,60.566925,-3.359925,True,False,False,safety kart lap
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7246,176,The Kartel,10866.080,53.703,53.8616,0.997055,1.007094,54.243669,-0.540669,False,False,False,green flag lap
7247,183,DNH Miles-Shelby Squad,10867.225,52.512,52.5783,0.998739,1.007409,52.967848,-0.455848,False,False,False,green flag lap
7248,182,BRITISH SOUTH AMERICAN RACING,10867.709,50.436,49.7402,1.013989,1.007409,50.108721,0.327279,False,False,False,green flag lap
7249,177,Dial In SODI,10869.790,53.713,53.6600,1.000988,1.007409,54.057563,-0.344563,False,False,False,green flag lap


In [38]:
# Lap-time ticks at readable motorsport intervals - a bare log axis renders "5" for
# 50s and again for 500s, which cannot be read.
_LAPTIME_TICKS = [45, 60, 90, 120, 180, 300, 600]
_LAPTIME_TICKTEXT = ["45s", "1:00", "1:30", "2:00", "3:00", "5:00", "10:00"]


def plot_lap_states(laps: pd.DataFrame, title: str | None = None) -> go.Figure:
    """Lap time vs race time, coloured by flag condition, shaped by pit / racing.

    Four buckets on two hues: colour carries the race-wide flag condition and
    marker shape the individual stop, so the eye never has to separate four
    colours at once.  Timing glitches are left out - they are not a race state,
    and a 14s lap stretches a log axis over a decade that holds no data - but they
    stay in `laps` and are counted in `lap_state_summary`.
    """
    df = laps[~laps["glitch"]].assign(
        race_time_min=lambda d: d["lap_end_s"] / 60,
        flag=lambda d: np.where(d["safety"], "safety kart", "green flag"),
        lap_type=lambda d: np.where(d["pit"], "pitstop", "racing"),
    )
    dropped = int(laps["glitch"].sum())
    fig = px.scatter(
        df,
        x="race_time_min",
        y="laptime_s",
        color="flag",
        symbol="lap_type",
        log_y=True,
        category_orders={
            "flag": ["green flag", "safety kart"],
            "lap_type": ["racing", "pitstop"],
        },
        color_discrete_map={"green flag": "#2a78d6", "safety kart": "#eb6834"},
        symbol_map={"racing": "circle", "pitstop": "diamond"},
        hover_name="competitor",
        hover_data={
            "lap": True,
            "state": True,
            "laptime_s": ":.2f",
            "excess_s": ":.1f",
            "race_time_min": ":.1f",
        },
        title=title or "Lap time by race condition",
        subtitle=f"{len(df):,} laps"
        + (f" · {dropped} timing glitches excluded" if dropped else ""),
    )
    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.update_traces(
        marker=dict(size=7, opacity=0.9), selector=lambda t: "pitstop" in t.name
    )
    fig.update_layout(
        template="plotly_white",
        plot_bgcolor="#fcfcfb",
        paper_bgcolor="#fcfcfb",
        font=dict(color="#52514e"),
        title=dict(font=dict(color="#0b0b0b")),
        legend=dict(title_text="", itemsizing="constant"),
        margin=dict(t=80, r=20),
    )
    fig.update_xaxes(title="Race time (min)", gridcolor="#e1e0d9", zeroline=False)
    fig.update_yaxes(
        title="Lap time",
        gridcolor="#e1e0d9",
        zeroline=False,
        tickmode="array",
        tickvals=_LAPTIME_TICKS,
        ticktext=_LAPTIME_TICKTEXT,
    )
    return fig


plot_lap_states(laps.loc[lambda d: d["competitor"].isin(laptimes.columns.tolist()[:4])], "Lap time by race condition — 2026 Jul Daytona SP 3h")

In [10]:
px.bar(start_times.dt.total_seconds())

In [3]:
sd_3hr.competitor_data(1)

,Lap,Pos,Lap Time,Diff to Last Lap,Diff to Best Lap,Gap in Front,Diff to P1,Speed
0,1,6,0 days 00:00:57.350000,0.000,10.444,0.674,3.204,56.495 km/h
1,2,6,0 days 00:01:11.865000,14.515,24.959,0.373,1.966,45.085 km/h
2,3,5,0 days 00:01:26.354000,14.489,39.448,0.476,2.096,37.520 km/h
3,4,5,0 days 00:01:30.862000,4.508,43.956,0.654,2.070,35.658 km/h
4,5,5,0 days 00:01:24.957000,-,38.051,0.256,1.414,38.137 km/h
...,...,...,...,...,...,...,...,...
194,195,1,0 days 00:00:47.622000,-,0.716,0.000,0.000,68.036 km/h
195,196,1,0 days 00:00:47.635000,0.013,0.729,0.000,0.000,68.017 km/h
196,197,1,0 days 00:00:48.834000,1.199,1.928,0.000,0.000,66.347 km/h
197,198,1,0 days 00:00:47.590000,-,0.684,0.000,0.000,68.082 km/h


In [ ]:
def plot_gap_to_mean_leader(laptimes: pd.DataFrame) -> go.Figure:
    leader_mean_lap = laptimes.iloc[:, 0].mean()
    return px.line(
        laptimes.assign(
                    **{
                        col: lambda d, col=col: (
                            (leader_mean_lap - d[col]).cumsum().dt.total_seconds()
                        )
                        for col in laptimes.columns
                    }
                )[laptimes.columns[:14]],
        title="Gap to leader mean lap time (cumulative) - 2026 Jul Dayona SP 3h",
    )


plot_gap_to_mean_leader(laptimes)

In [37]:
def plot_gap_to_mean_leader(
    timestamps: pd.DataFrame, laptimes: pd.DataFrame
) -> go.Figure:
    leader_mean_lap = laptimes.iloc[:, 0].mean()

    gap = laptimes.assign(
                **{
                    col: lambda d, col=col: (
                        (leader_mean_lap - d[col]).cumsum().dt.total_seconds()
                    )
                    for col in laptimes.columns
                }
            )
    display(gap)
    race_time = timestamps.stack().dt.total_seconds() / 60

    df = (
        pd.concat([race_time.rename("race_time_min"), gap.stack().rename("gap_s")], axis=1)
        .rename_axis(index=["lap", "competitor"])
        .reset_index()
    )

    fig = px.line(
        df,
        x="race_time_min",
        y="gap_s",
        color="competitor",
        hover_data=["lap"],
        title="Gap to leader mean lap time (cumulative) - 2026 Jul Daytona SP 3h",
    )
    fig.update_layout(
        xaxis_title="Race time (min)", yaxis_title="Gap (s)", legend_title=""
    )
    return fig


plot_gap_to_mean_leader(timestamps, laptimes)


,dendi239,The Mandem - JM,WPR,Billy No Mates,Clean Bulls Racing,DRS and Inshallah,TDS Racing,Maple Motorsport,Late Brakers,Ferrari LH44,...,Kizzler's Army,The Fat And The Curious,TH14 Racing,J J J Racing,DNH Hawthorn's Heroes,On Your Marks DEXET Go!,Houghton Racing,Shake 'n' Bake,Frenchies,Hashnet
0,-2.995050e+00,-2.116050,-1.564050,-4.239050,-4.629050,-4.920050,-1.685050,-466.408050,-2.287050,-5.504050,...,-21.627050,-8.614050,-274.711050,-9.183050,-21.450050,-21.080050,-22.164050,-21.247050,-20.124050,-6.385050
1,-2.050510e+01,-20.449101,-20.312101,-22.360101,-22.629101,-22.363101,-20.310101,-490.372101,-20.761101,-23.605101,...,-50.535101,-28.351101,-294.338101,-28.237101,-219.977101,-49.883101,-50.885101,-50.042101,-48.956101,-24.236101
2,-5.250415e+01,-181.036151,-52.181151,-53.733151,-54.067151,-54.200151,-52.240151,-483.920151,-52.235151,-55.170151,...,-76.096151,-345.874151,-336.775151,-56.519151,-238.764151,-75.594151,-75.894151,-75.940151,-75.023151,-53.608151
3,-8.901120e+01,-199.409201,-88.714201,-90.310201,-90.691201,-90.171201,-88.761201,-477.214201,-88.738201,-301.462201,...,-98.796201,-370.540201,-364.119201,-96.386201,-333.328201,-97.613201,-100.080201,-97.861201,-96.623201,-89.711201
4,-1.196133e+02,-241.499251,-119.972251,-120.703251,-121.167251,-120.421251,-119.914251,-471.292251,-119.759251,-325.087251,...,-125.281251,-369.597251,-365.440251,-126.542251,-423.448251,-124.389251,-125.321251,-124.724251,-123.689251,-119.908251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,-2.452980e+01,-119.581799,-120.881799,-177.209799,-189.920799,-195.356799,-216.153799,-233.330799,-237.733799,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
195,-1.780985e+01,-112.299849,-113.574849,-170.666849,-183.109849,-190.157849,-209.470849,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,-1.228890e+01,-105.182900,-106.690900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,-5.523950e+00,-99.055950,-99.920950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [233]:
def plot_laptime_distribution(laptimes: pd.DataFrame) -> go.Figure:
    best_lap = laptimes.where(lambda d: d > d.median().min() * 0.93).min().min()
    return (
        laptimes.where(lambda d: (d > best_lap * 0.93) & (d < best_lap * 1.07))
        .assign(
            **{
                col: lambda d, c=col: d[c].dt.total_seconds()
                for col in laptimes.columns
            }
        )
        .pipe(px.histogram, marginal="box", nbins=30, barmode="overlay")
    )


plot_laptime_distribution(laptimes)